# Relational Databases & Normalization 

This notebook is the SQL-adjacent side of relational database theory: before writing real SQL queries, you need to understand *why* tables are structured the way they are. That "why" is **normalization** — the process of removing data redundancy.

We'll use real SQL (via Python's built-in `sqlite3`) to see a **bad, unnormalized table**, understand exactly what rules it breaks, and then fix it step by step into 1NF and 2NF and 3NF.

## Background: Flat Files vs. Relational Databases

A **Flat File** stores data in one simple structure with no relationships between separate tables (Session 1). This is fine for small, simple data — but it struggles with two things:

- **Integrity** (یکپارچگی) — keeping facts accurate as data changes
- **Consistency** (استحکام) — making sure the same fact isn't stored in two places and allowed to disagree with itself

In **June 1970**, Edgar F. Codd (IBM) published *"A Relational Model of Data for Large Shared Data Banks,"* introducing the **Relational Database** model — data organized into linked tables — specifically to solve these flat-file problems. Codd's later papers introduced **Normalization** and the first three **Normal Forms**.

A **Database** (پایگاه داده) is simply defined as: a collection of organized data.

## Step 1: A Bad, Unnormalized Table

Let's set up a single flat table for a course enrollment system — the kind of table you'd get if you just dumped everything into one spreadsheet without thinking about structure.

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

# An UNNORMALIZED table: student + course + instructor info all mixed together
cur.execute('''
CREATE TABLE Enrollments_Bad (
    StudentID INTEGER,
    StudentName TEXT,
    CourseID TEXT,
    CourseName TEXT,
    Instructor TEXT,
    Grade TEXT
)
''')

rows = [
    (1, "Ali",    "CS101", "Intro to Programming", "Dr. Smith", "A"),
    (1, "Ali",    "MA201", "Linear Algebra",        "Dr. Jones", "B"),
    (2, "Sara",   "CS101", "Intro to Programming", "Dr. Smith", "A"),
    (3, "Behnam", "CS101", "Intro to Programming", "Dr. Smith", "B"),
]
cur.executemany("INSERT INTO Enrollments_Bad VALUES (?, ?, ?, ?, ?, ?)", rows)
conn.commit()

pd.read_sql("SELECT * FROM Enrollments_Bad", conn)

,StudentID,StudentName,CourseID,CourseName,Instructor,Grade
0,1,Ali,CS101,Intro to Programming,Dr. Smith,A
1,1,Ali,MA201,Linear Algebra,Dr. Jones,B
2,2,Sara,CS101,Intro to Programming,Dr. Smith,A
3,3,Behnam,CS101,Intro to Programming,Dr. Smith,B


### Spotting the Problem: Redundancy

Notice that `"CS101", "Intro to Programming", "Dr. Smith"` is repeated three times. This is **Redundancy** (افزونگی داده) — the same fact (which instructor teaches CS101) is stored in multiple rows.

Why is this bad?

- If Dr. Smith is replaced by a new instructor, you'd have to update **every row** that mentions CS101 — miss one, and now the data **contradicts itself**. This is exactly the Consistency problem Codd's model was built to prevent.
- It also wastes storage repeating the same course name and instructor over and over.

Let's check this with SQL directly:

In [2]:
# Prove the redundancy: how many times is each CourseID's info repeated?
pd.read_sql('''
    SELECT CourseID, CourseName, Instructor, COUNT(*) AS times_repeated
    FROM Enrollments_Bad
    GROUP BY CourseID
''', conn)

,CourseID,CourseName,Instructor,times_repeated
0,CS101,Intro to Programming,Dr. Smith,3
1,MA201,Linear Algebra,Dr. Jones,1


## Step 2: Checking 1NF

Recall the three 1NF rules:

1. Every row must be unique
2. Every value in a column must share the same data type
3. Every field must hold a single, atomic value (no multiple values crammed into one field)

Our `Enrollments_Bad` table actually *does* satisfy 1NF — every row is unique, every column has a consistent type, and no field holds multiple values. **1NF is about row/field shape, not about redundancy across rows** — that's what 2NF exists to catch.

To make the *violation* of 1NF concrete, here's what a rule-3 violation would look like:

In [3]:
# Example of a 1NF VIOLATION: a field holding multiple values at once
bad_1nf_example = pd.DataFrame({
    "StudentID": [1],
    "StudentName": ["Ali"],
    "Courses": ["CS101, MA201, PHY101"],   # <-- not atomic: three values crammed into one field
})
bad_1nf_example

,StudentID,StudentName,Courses
0,1,Ali,"CS101, MA201, PHY101"


This `Courses` field breaks 1NF's third rule directly — you can't easily filter, count, or join on individual courses when they're bundled into a single string. The fix is to give each (student, course) pair its own row, which is exactly what `Enrollments_Bad` already does correctly.

## Step 3: Applying 2NF — Splitting by Functional Dependency

2NF says: **every non-key column must depend on the whole key, not just part of it.**

In `Enrollments_Bad`, the natural key is the pair `(StudentID, CourseID)` — together they identify one enrollment. But look at the functional dependencies:

```text
CourseID  ══════>  CourseName, Instructor      (depends only on CourseID, not on StudentID too)
StudentID ══════>  StudentName                 (depends only on StudentID, not on CourseID too)
```

Both of these are **partial dependencies** — `CourseName` and `Instructor` don't need `StudentID` at all to be determined, and `StudentName` doesn't need `CourseID`. This is precisely what causes the redundancy we found above. The fix: split the table into three, one per functional dependency.

In [4]:
# Table 1: Students (StudentID -> StudentName)
cur.execute('''
CREATE TABLE Students (
    StudentID INTEGER PRIMARY KEY,
    StudentName TEXT NOT NULL
)
''')
cur.executemany("INSERT INTO Students VALUES (?, ?)", [
    (1, "Ali"), (2, "Sara"), (3, "Behnam")
])

# Table 2: Courses (CourseID -> CourseName, Instructor)
cur.execute('''
CREATE TABLE Courses (
    CourseID TEXT PRIMARY KEY,
    CourseName TEXT NOT NULL,
    Instructor TEXT NOT NULL
)
''')
cur.executemany("INSERT INTO Courses VALUES (?, ?, ?)", [
    ("CS101", "Intro to Programming", "Dr. Smith"),
    ("MA201", "Linear Algebra", "Dr. Jones"),
])

# Table 3: Enrollments ((StudentID, CourseID) -> Grade)
cur.execute('''
CREATE TABLE Enrollments (
    StudentID INTEGER,
    CourseID TEXT,
    Grade TEXT,
    PRIMARY KEY (StudentID, CourseID),
    FOREIGN KEY (StudentID) REFERENCES Students(StudentID),
    FOREIGN KEY (CourseID) REFERENCES Courses(CourseID)
)
''')
cur.executemany("INSERT INTO Enrollments VALUES (?, ?, ?)", [
    (1, "CS101", "A"),
    (1, "MA201", "B"),
    (2, "CS101", "A"),
    (3, "CS101", "B"),
])
conn.commit()
print("Three normalized tables created: Students, Courses, Enrollments")

Three normalized tables created: Students, Courses, Enrollments


In [5]:
print("Students:")
display(pd.read_sql("SELECT * FROM Students", conn))

print("\nCourses:")
display(pd.read_sql("SELECT * FROM Courses", conn))

print("\nEnrollments:")
display(pd.read_sql("SELECT * FROM Enrollments", conn))

Students:


,StudentID,StudentName
0,1,Ali
1,2,Sara
2,3,Behnam



Courses:


,CourseID,CourseName,Instructor
0,CS101,Intro to Programming,Dr. Smith
1,MA201,Linear Algebra,Dr. Jones



Enrollments:


,StudentID,CourseID,Grade
0,1,CS101,A
1,1,MA201,B
2,2,CS101,A
3,3,CS101,B


Notice: `"CS101", "Intro to Programming", "Dr. Smith"` now appears **exactly once**, in the `Courses` table — no matter how many students enroll in it. The redundancy is gone. If Dr. Smith is replaced, you update **one row**, in one table, and it's correct everywhere.

## 3NF (Third Normal Form)

Recall the sequence so far:

- **1NF** — fixes the *shape* of a single table (unique rows, consistent types, atomic fields)
- **2NF** — fixes *partial* dependency (a non-key column depending on only part of a composite key)
- **3NF** — fixes **transitive dependency**: a non-key column depending on *another non-key column*, instead of depending directly on the key

**Rule:** A table is in 3NF when it's already in 2NF, and every non-key column depends **only** on the key — not on some other non-key column.


### Example: A Table That Violates 3NF

Consider a student table that also stores department information:

In [8]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

cur.execute('''
CREATE TABLE Students_Bad3NF (
    StudentID INTEGER PRIMARY KEY,
    StudentName TEXT,
    Department TEXT,
    DepartmentBuilding TEXT
)
''')
cur.executemany("INSERT INTO Students_Bad3NF VALUES (?, ?, ?, ?)", [
    (1, "Ali",    "Computer Science", "Tech Building"),
    (2, "Sara",   "Computer Science", "Tech Building"),
    (3, "Behnam", "Mathematics",      "Science Hall"),
])
conn.commit()

pd.read_sql("SELECT * FROM Students_Bad3NF", conn)

,StudentID,StudentName,Department,DepartmentBuilding
0,1,Ali,Computer Science,Tech Building
1,2,Sara,Computer Science,Tech Building
2,3,Behnam,Mathematics,Science Hall


### Why This Violates 3NF

Trace the dependencies:

```text
StudentID  ══════>  Department            (fine — depends on the key)
Department ══════>  DepartmentBuilding    (this is the problem)
```

`DepartmentBuilding` doesn't actually depend on `StudentID` — it depends on `Department`, which is itself just a non-key column. This is a **transitive dependency**: `StudentID → Department → DepartmentBuilding`.

The redundancy this causes is visible directly: `"Computer Science", "Tech Building"` is repeated for every student in that department. If the CS department relocates, you'd need to update every matching row — the same consistency risk 2NF was designed to prevent, just one link further down the chain.

In [9]:
# Fix: split off Department into its own table
cur.execute('''
CREATE TABLE Departments (
    Department TEXT PRIMARY KEY,
    Building TEXT NOT NULL
)
''')
cur.executemany("INSERT INTO Departments VALUES (?, ?)", [
    ("Computer Science", "Tech Building"),
    ("Mathematics",      "Science Hall"),
])

cur.execute('''
CREATE TABLE Students_3NF (
    StudentID INTEGER PRIMARY KEY,
    StudentName TEXT,
    Department TEXT,
    FOREIGN KEY (Department) REFERENCES Departments(Department)
)
''')
cur.executemany("INSERT INTO Students_3NF VALUES (?, ?, ?)", [
    (1, "Ali",    "Computer Science"),
    (2, "Sara",   "Computer Science"),
    (3, "Behnam", "Mathematics"),
])
conn.commit()

print("Students_3NF:")
display(pd.read_sql("SELECT * FROM Students_3NF", conn))
print("\nDepartments:")
display(pd.read_sql("SELECT * FROM Departments", conn))

Students_3NF:


,StudentID,StudentName,Department
0,1,Ali,Computer Science
1,2,Sara,Computer Science
2,3,Behnam,Mathematics



Departments:


,Department,Building
0,Computer Science,Tech Building
1,Mathematics,Science Hall


Now `"Tech Building"` is stored exactly once, in `Departments`. `Students_3NF` only stores which department a student belongs to — the building is looked up via a `JOIN` whenever needed:

In [10]:
pd.read_sql('''
    SELECT s.StudentName, s.Department, d.Building
    FROM Students_3NF s
    JOIN Departments d ON s.Department = d.Department
''', conn)

,StudentName,Department,Building
0,Ali,Computer Science,Tech Building
1,Sara,Computer Science,Tech Building
2,Behnam,Mathematics,Science Hall


### Quick Reference: All Three Normal Forms

| Form | Problem It Fixes | Rule |
|---|---|---|
| **1NF** | Messy table shape | Unique rows, consistent column types, atomic (single-value) fields |
| **2NF** | Partial dependency | Every non-key column depends on the **whole** key, not part of it |
| **3NF** | Transitive dependency | Every non-key column depends **only** on the key, not on another non-key column |

---


## Table Relationships: One-to-Many, Many-to-One, Many-to-Many

Once tables are normalized and split apart, they need to be connected back together — and *how* they connect falls into one of three relationship types. These aren't independent concepts from normalization; they're a direct consequence of it. Splitting `Students_Bad3NF` above already created a relationship between `Students_3NF` and `Departments`.

### 1. One-to-Many (and Many-to-One — Same Relationship, Two Directions)

**One-to-Many** and **Many-to-One** describe the *exact same relationship*, just viewed from opposite sides:

- "**One** Department has **Many** Students" → One-to-Many, viewed from Departments
- "**Many** Students belong to **One** Department" → Many-to-One, viewed from Students

This is precisely the `Departments` ↔ `Students_3NF` relationship above. The rule for implementing it in SQL: **put the foreign key on the "many" side.**

```text
Departments (one)  ◄──────  Students_3NF (many)
   Department (PK)              Department (FK)
```

Let's confirm with a query — one department, many students:

In [11]:
pd.read_sql('''
    SELECT d.Department, d.Building, COUNT(s.StudentID) AS num_students
    FROM Departments d
    LEFT JOIN Students_3NF s ON d.Department = s.Department
    GROUP BY d.Department
''', conn)

,Department,Building,num_students
0,Computer Science,Tech Building,2
1,Mathematics,Science Hall,1


Each department here has more than one student, but each student belongs to exactly one department — that asymmetry is the definition of One-to-Many / Many-to-One.

### More One-to-Many Examples (for pattern recognition)

| "One" side | "Many" side | Real-world relationship |
|---|---|---|
| Customer | Orders | One customer places many orders |
| Author | Books | One author can write many books |
| Manager | Employees | One manager supervises many employees |
| Country | Cities | One country contains many cities |

In every case, the foreign key goes on the "many" table (`Orders.CustomerID`, `Books.AuthorID`, `Employees.ManagerID`, `Cities.CountryID`).

### 2. Many-to-Many

**Many-to-Many** is when records on *both* sides can relate to multiple records on the other side. The clearest example is Students and Courses — this is exactly the `Enrollments` table from the last session:

- One student can take **many** courses
- One course can have **many** students enrolled

Neither side can hold a simple foreign key pointing to the other, because a single column can't reference multiple rows at once. The fix is a **junction table** (also called a bridge or associative table) sitting between the two, holding one row per actual pairing.

In [12]:
cur.execute('''
CREATE TABLE Courses (
    CourseID TEXT PRIMARY KEY,
    CourseName TEXT NOT NULL
)
''')
cur.executemany("INSERT INTO Courses VALUES (?, ?)", [
    ("CS101", "Intro to Programming"),
    ("MA201", "Linear Algebra"),
    ("PHY101", "Physics I"),
])

# The junction table: one row per (student, course) pairing
cur.execute('''
CREATE TABLE Enrollments (
    StudentID INTEGER,
    CourseID TEXT,
    PRIMARY KEY (StudentID, CourseID),
    FOREIGN KEY (StudentID) REFERENCES Students_3NF(StudentID),
    FOREIGN KEY (CourseID) REFERENCES Courses(CourseID)
)
''')
cur.executemany("INSERT INTO Enrollments VALUES (?, ?)", [
    (1, "CS101"), (1, "MA201"), (1, "PHY101"),   # Ali takes 3 courses
    (2, "CS101"),                                  # Sara takes 1 course
    (3, "CS101"), (3, "MA201"),                    # Behnam takes 2 courses
])
conn.commit()

print("Courses:")
display(pd.read_sql("SELECT * FROM Courses", conn))
print("\nEnrollments (the junction table):")
display(pd.read_sql("SELECT * FROM Enrollments", conn))

Courses:


,CourseID,CourseName
0,CS101,Intro to Programming
1,MA201,Linear Algebra
2,PHY101,Physics I



Enrollments (the junction table):


,StudentID,CourseID
0,1,CS101
1,1,MA201
2,1,PHY101
3,2,CS101
4,3,CS101
5,3,MA201


In [13]:
# Proving it's Many-to-Many: each student has multiple courses, AND each course has multiple students
print("Courses per student:")
display(pd.read_sql('''
    SELECT s.StudentName, COUNT(e.CourseID) AS num_courses
    FROM Students_3NF s JOIN Enrollments e ON s.StudentID = e.StudentID
    GROUP BY s.StudentName
''', conn))

print("\nStudents per course:")
display(pd.read_sql('''
    SELECT c.CourseName, COUNT(e.StudentID) AS num_students
    FROM Courses c JOIN Enrollments e ON c.CourseID = e.CourseID
    GROUP BY c.CourseName
''', conn))

Courses per student:


,StudentName,num_courses
0,Ali,3
1,Behnam,2
2,Sara,1



Students per course:


,CourseName,num_students
0,Intro to Programming,3
1,Linear Algebra,2
2,Physics I,1


Both counts exceed 1 in places — confirming the relationship genuinely goes both directions, unlike One-to-Many where only one side could have counts greater than 1.

### More Many-to-Many Examples (for pattern recognition)

| Table A | Table B | Junction Table | Real-world relationship |
|---|---|---|---|
| Students | Courses | Enrollments | A student takes many courses; a course has many students |
| Authors | Books | BookAuthors | A book can have multiple authors; an author writes multiple books |
| Actors | Movies | MovieCast | A movie has many actors; an actor appears in many movies |
| Products | Orders | OrderItems | An order contains many products; a product appears in many orders |

The pattern is always the same: **whenever "many" appears on both sides of a relationship, you need a junction table** — a plain foreign key on either side alone can't represent it.

---

## Summary Diagram: All Three Relationship Types

```text
ONE-TO-MANY / MANY-TO-ONE:
   Departments (1) ─────────< Students (many)
      [foreign key lives on the "many" side]

MANY-TO-MANY:
   Students (many) >──── Enrollments ────< Courses (many)
      [a junction table sits in between, holding foreign keys to both]
```


## Normalization — Step by Step, From Scratch

### Step 1: Start With a Flat, Unnormalized Table

Look at your raw table and ask: does the same information repeat across multiple rows unnecessarily? If yes, it needs normalizing.

**Example — the raw table:**

| StudentID | StudentName | CourseID | CourseName | Instructor | Department | Building |
|---|---|---|---|---|---|---|
| 1 | Ali | CS101 | Intro to Programming | Dr. Smith | CS | Tech Bldg |
| 1 | Ali | MA201 | Linear Algebra | Dr. Jones | CS | Tech Bldg |
| 2 | Sara | CS101 | Intro to Programming | Dr. Smith | CS | Tech Bldg |

### Step 2: Identify the Key Column vs. Non-Key Columns

This is the part you asked about specifically — here's how to actually tell them apart:

**A Key Column is whichever column (or combination of columns) uniquely identifies one row.** Ask: *"If I know only this value, do I know exactly which row I'm talking about?"*

- `StudentID` alone doesn't uniquely identify a row here (Ali has two rows). Neither does `CourseID` alone.
- But the **pair** `(StudentID, CourseID)` together does — no two rows share that same combination. So the key here is the composite key `(StudentID, CourseID)`.

**Everything else is a Non-Key Column** — every column *not* part of that identifying combination: `StudentName`, `CourseName`, `Instructor`, `Department`, `Building`.

**The practical test, column by column:**
1. Can this column repeat for different real-world entities? → it's likely a non-key column.
2. Does this column (alone or combined with others) never repeat, and every other column can be looked up from it? → it's a key column.
3. Write out the dependencies as arrows: `KeyColumn → NonKeyColumn`. If a non-key column's value is fully determined once you know the key, that arrow is valid — this is a **Functional Dependency**.

### Step 3: Apply 1NF

**Rule:** unique rows + consistent data type per column + atomic (single-value) fields.

Check the table above — no field holds multiple values, every row is unique, columns are consistent types. This table already passes 1NF. (A 1NF *violation* would look like a `Courses` field containing `"CS101, MA201"` crammed into one cell — that needs splitting into separate rows first.)

### Step 4: Apply 2NF — Remove Partial Dependency

**Rule:** every non-key column must depend on the **whole** key, not just part of it.

Our key is `(StudentID, CourseID)`. Check each non-key column:

```text
StudentName  ← depends only on StudentID (not CourseID)   → PARTIAL dependency
CourseName   ← depends only on CourseID (not StudentID)   → PARTIAL dependency
Instructor   ← depends only on CourseID (not StudentID)   → PARTIAL dependency
```

Since these depend on only *part* of the composite key, they violate 2NF. **Fix:** split them into separate tables, one per partial dependency:

- `Students(StudentID, StudentName)`
- `Courses(CourseID, CourseName, Instructor, Department, Building)`
- `Enrollments(StudentID, CourseID)` ← what's left, the pure relationship

### Step 5: Apply 3NF — Remove Transitive Dependency

**Rule:** every non-key column must depend **only** on the key — not on another non-key column.

Look inside the new `Courses` table. Its key is `CourseID`. Check the remaining columns:

```text
CourseID → Department            (fine — depends on the key)
Department → Building            (PROBLEM — depends on another non-key column)
```

`Building` doesn't depend on `CourseID` directly — it depends on `Department`, which is itself non-key. This chain, `CourseID → Department → Building`, is a **transitive dependency**. **Fix:** split it out:

- `Courses(CourseID, CourseName, Instructor, Department)`
- `Departments(Department, Building)`

### Final Normalized Schema

```text
Students(StudentID, StudentName)
Departments(Department, Building)
Courses(CourseID, CourseName, Instructor, Department)
Enrollments(StudentID, CourseID)
```

Each fact now lives in exactly one place.

---

## Relationships — How to Identify Which Type Applies

Once tables are split, ask this for **each pair** of tables: *"Can one row on Side A relate to multiple rows on Side B — and vice versa?"*

| Ask this | Answer | Relationship | How to implement |
|---|---|---|---|
| Can one Department have many Students, but each Student belongs to only one Department? | Yes, one-directional "many" | **One-to-Many / Many-to-One** | Put a foreign key on the "many" side (`Students.Department`) |
| Can one Student take many Courses, **and** one Course have many Students? | Yes, "many" on **both** sides | **Many-to-Many** | Create a junction table with foreign keys to both (`Enrollments(StudentID, CourseID)`) |

**Applied to our example:**

- `Departments` ↔ `Courses` → One department has many courses, but each course belongs to one department → **One-to-Many** (foreign key `Courses.Department`)
- `Students` ↔ `Courses` → many students per course AND many courses per student → **Many-to-Many** → needs the `Enrollments` junction table

**The quick rule of thumb:** if a single foreign key column can express the relationship, it's One-to-Many/Many-to-One. If you find yourself needing a foreign key that would have to hold *multiple* values, that's your signal you actually need a junction table — Many-to-Many.

## 1. The Problem: One Flat Table

Here is the raw data, exactly as it would appear in a single spreadsheet — one row per product on an order.

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

cur.execute('''
CREATE TABLE Flat (
    OrderCode INTEGER,
    OrderLineCode INTEGER,
    ProductCode INTEGER,
    UnitPrice INTEGER,
    OrderQuantity INTEGER,
    TaxAmount REAL,
    Discount REAL,
    ProductName TEXT,
    CustomerCode INTEGER,
    CustomerName TEXT,
    EmployeeCode INTEGER,
    EmployeeName TEXT,
    OrderDate TEXT,
    ProductCategoryCode INTEGER,
    ProductCategoryName TEXT
)
''')
cur.executemany("INSERT INTO Flat VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)", [
    (100, 1, 500, 1000,  1, 0.1, 0.01, "ProductA", 555, "Ali Rahmani",     666, "Behnam Shaygan", "1405-05-05", 1, "CategoryA"),
    (100, 2, 600, 4000,  2, 0,   0,    "ProductM", 555, "Ali Rahmani",     666, "Behnam Shaygan", "1405-05-05", 2, "CategoryM"),
    (100, 3, 700, 80000, 1, 0.2, 0.02, "ProductO", 555, "Ali Rahmani",     666, "Behnam Shaygan", "1405-05-05", 3, "CategoryO"),
    (101, 1, 700, 80000, 3, 0.2, 0.02, "ProductO", 556, "Fatemeh Rezaeie", 666, "Behnam Shaygan", "1405-05-06", 3, "CategoryO"),
])
conn.commit()

pd.read_sql("SELECT * FROM Flat", conn)

,OrderCode,OrderLineCode,ProductCode,UnitPrice,OrderQuantity,TaxAmount,Discount,ProductName,CustomerCode,CustomerName,EmployeeCode,EmployeeName,OrderDate,ProductCategoryCode,ProductCategoryName
0,100,1,500,1000,1,0.1,0.01,ProductA,555,Ali Rahmani,666,Behnam Shaygan,1405-05-05,1,CategoryA
1,100,2,600,4000,2,0.0,0.00,ProductM,555,Ali Rahmani,666,Behnam Shaygan,1405-05-05,2,CategoryM
2,100,3,700,80000,1,0.2,0.02,ProductO,555,Ali Rahmani,666,Behnam Shaygan,1405-05-05,3,CategoryO
3,101,1,700,80000,3,0.2,0.02,ProductO,556,Fatemeh Rezaeie,666,Behnam Shaygan,1405-05-06,3,CategoryO


### Proving the Redundancy With SQL

Rather than just eyeballing it, let's confirm the repetition directly: how many times does the same customer, employee, and product information actually repeat?

In [3]:
print("Customer info repeated per CustomerCode:")
display(pd.read_sql('''
    SELECT CustomerCode, CustomerName, COUNT(*) AS times_repeated
    FROM Flat GROUP BY CustomerCode
''', conn))

print("\nProduct info repeated per ProductCode:")
display(pd.read_sql('''
    SELECT ProductCode, ProductName, UnitPrice, COUNT(*) AS times_repeated
    FROM Flat GROUP BY ProductCode
''', conn))

Customer info repeated per CustomerCode:


,CustomerCode,CustomerName,times_repeated
0,555,Ali Rahmani,3
1,556,Fatemeh Rezaeie,1



Product info repeated per ProductCode:


,ProductCode,ProductName,UnitPrice,times_repeated
0,500,ProductA,1000,1
1,600,ProductM,4000,1
2,700,ProductO,80000,2


`ProductO` (code 700) is stored twice, with the identical name and price both times — pure redundancy. This is exactly the problem Normalization exists to remove.

## 2. The Key Step: Deciding What Each Column Belongs To

For every column, ask: **"What entity does this information actually describe?"**

| Columns | Belongs To |
|---|---|
| `OrderCode`, `OrderDate`, `CustomerCode`, `EmployeeCode` | **Order** (the order as a whole) |
| `OrderCode`, `OrderLineCode`, `ProductCode`, `OrderQuantity`, `TaxAmount`, `Discount` | **Order Line** (one product within one order) |
| `ProductCode`, `ProductName`, `UnitPrice`, `ProductCategoryCode` | **Product** |
| `ProductCategoryCode`, `ProductCategoryName` | **Category** |
| `CustomerCode`, `CustomerName` | **Customer** |
| `EmployeeCode`, `EmployeeName` | **Employee** |

Six groups → six tables. Let's build them all.

## 3. Building the Tables

In [4]:
# Customer — one row per customer
cur.execute('''
CREATE TABLE Customer (
    CustomerCode INTEGER PRIMARY KEY,
    CustomerName TEXT NOT NULL
)
''')
cur.executemany("INSERT INTO Customer VALUES (?, ?)", [
    (555, "Ali Rahmani"),
    (556, "Fatemeh Rezaeie"),
])

# Employee — one row per employee
cur.execute('''
CREATE TABLE Employee (
    EmployeeCode INTEGER PRIMARY KEY,
    EmployeeName TEXT NOT NULL
)
''')
cur.executemany("INSERT INTO Employee VALUES (?, ?)", [
    (666, "Behnam Shaygan"),
])

# ProductCategory — one row per category
cur.execute('''
CREATE TABLE ProductCategory (
    ProductCategoryCode INTEGER PRIMARY KEY,
    ProductCategoryName TEXT NOT NULL
)
''')
cur.executemany("INSERT INTO ProductCategory VALUES (?, ?)", [
    (1, "CategoryA"),
    (2, "CategoryM"),
    (3, "CategoryO"),
])

conn.commit()
print("Customer, Employee, ProductCategory created")

Customer, Employee, ProductCategory created


In [5]:
# Product — references ProductCategory via a Foreign Key
cur.execute('''
CREATE TABLE Product (
    ProductCode INTEGER PRIMARY KEY,
    ProductName TEXT NOT NULL,
    UnitPrice INTEGER NOT NULL,
    ProductCategoryCode INTEGER,
    FOREIGN KEY (ProductCategoryCode) REFERENCES ProductCategory(ProductCategoryCode)
)
''')
cur.executemany("INSERT INTO Product VALUES (?, ?, ?, ?)", [
    (500, "ProductA", 1000,  1),
    (600, "ProductM", 4000,  2),
    (700, "ProductO", 80000, 3),
])
conn.commit()
print("Product created — ProductCategoryCode is a Foreign Key pointing to ProductCategory")

Product created — ProductCategoryCode is a Foreign Key pointing to ProductCategory


In [6]:
# Orders — the order "header": references Customer and Employee via Foreign Keys
cur.execute('''
CREATE TABLE Orders (
    OrderCode INTEGER PRIMARY KEY,
    OrderDate TEXT NOT NULL,
    CustomerCode INTEGER,
    EmployeeCode INTEGER,
    FOREIGN KEY (CustomerCode) REFERENCES Customer(CustomerCode),
    FOREIGN KEY (EmployeeCode) REFERENCES Employee(EmployeeCode)
)
''')
cur.executemany("INSERT INTO Orders VALUES (?, ?, ?, ?)", [
    (100, "1405-05-05", 555, 666),
    (101, "1405-05-06", 556, 666),
])
conn.commit()
print("Orders created — CustomerCode and EmployeeCode are both Foreign Keys")

Orders created — CustomerCode and EmployeeCode are both Foreign Keys


### Why `OrderDetails` Needs a Composite Key

Order 100 has three lines. If we tried to use `OrderCode` alone as the key, all three rows would collide — `100` isn't unique on its own:

In [7]:
# Demonstrating the problem: OrderCode alone repeats
pd.read_sql("SELECT OrderCode, OrderLineCode, ProductCode FROM Flat WHERE OrderCode = 100", conn)

,OrderCode,OrderLineCode,ProductCode
0,100,1,500
1,100,2,600
2,100,3,700


`OrderCode = 100` appears three times, but each `(OrderCode, OrderLineCode)` pair is unique. So the key for `OrderDetails` has to be the **combination** of both columns — a **Composite Primary Key**.

In [8]:
# OrderDetails — the order "lines": composite primary key, and TWO foreign keys
cur.execute('''
CREATE TABLE OrderDetails (
    OrderCode INTEGER,
    OrderLineCode INTEGER,
    ProductCode INTEGER,
    Quantity INTEGER,
    TaxAmount REAL,
    Discount REAL,
    PRIMARY KEY (OrderCode, OrderLineCode),
    FOREIGN KEY (OrderCode) REFERENCES Orders(OrderCode),
    FOREIGN KEY (ProductCode) REFERENCES Product(ProductCode)
)
''')
cur.executemany("INSERT INTO OrderDetails VALUES (?, ?, ?, ?, ?, ?)", [
    (100, 1, 500, 1, 0.1, 0.01),
    (100, 2, 600, 2, 0,   0),
    (100, 3, 700, 1, 0.2, 0.02),
    (101, 1, 700, 3, 0.2, 0.02),
])
conn.commit()
print("OrderDetails created — composite key (OrderCode, OrderLineCode)")
print("Two foreign keys: OrderCode -> Orders, ProductCode -> Product")

OrderDetails created — composite key (OrderCode, OrderLineCode)
Two foreign keys: OrderCode -> Orders, ProductCode -> Product


## 4. The Final Schema, All Six Tables

In [9]:
for table in ["Customer", "Employee", "ProductCategory", "Product", "Orders", "OrderDetails"]:
    print(f"{table}:")
    display(pd.read_sql(f"SELECT * FROM {table}", conn))
    print()

Customer:


,CustomerCode,CustomerName
0,555,Ali Rahmani
1,556,Fatemeh Rezaeie



Employee:


,EmployeeCode,EmployeeName
0,666,Behnam Shaygan



ProductCategory:


...

,ProductCode,ProductName,UnitPrice,ProductCategoryCode
0,500,ProductA,1000,1
1,600,ProductM,4000,2
2,700,ProductO,80000,3



Orders:


,OrderCode,OrderDate,CustomerCode,EmployeeCode
0,100,1405-05-05,555,666
1,101,1405-05-06,556,666



OrderDetails:


,OrderCode,OrderLineCode,ProductCode,Quantity,TaxAmount,Discount
0,100,1,500,1,0.1,0.01
1,100,2,600,2,0.0,0.00
2,100,3,700,1,0.2,0.02
3,101,1,700,3,0.2,0.02


## 5. Finding the Relationships and Their Cardinality

For every pair of tables connected by a foreign key, ask the same question:

> **Can one row on Side A match many rows on Side B — or only one?**

Rather than answering by inspection, let's prove each answer with an actual SQL query.

### Relationship 1: Customer (1) → Orders (\*)

In [10]:
pd.read_sql('''
    SELECT c.CustomerCode, c.CustomerName, COUNT(o.OrderCode) AS num_orders
    FROM Customer c
    LEFT JOIN Orders o ON c.CustomerCode = o.CustomerCode
    GROUP BY c.CustomerCode
''', conn)

,CustomerCode,CustomerName,num_orders
0,555,Ali Rahmani,1
1,556,Fatemeh Rezaeie,1


Even with only one order each in this small dataset, the *design* allows `num_orders` to be any number — that's what makes it One-to-Many, not that today's data happens to show more than one.

### Relationship 2: Employee (1) → Orders (\*)

In [13]:
pd.read_sql('''
    SELECT e.EmployeeCode, e.EmployeeName, COUNT(o.OrderCode) AS num_orders
    FROM Employee e
    LEFT JOIN Orders o ON e.EmployeeCode = o.EmployeeCode
    GROUP BY e.EmployeeCode
''', conn)

,EmployeeCode,EmployeeName,num_orders
0,666,Behnam Shaygan,2


Here it's visible directly: Behnam Shaygan (one employee) is linked to **2** orders — proving the "many" side of `Employee 1 ─── * Orders`.

### Relationship 3: ProductCategory (1) → Product (\*)

In [14]:
pd.read_sql('''
    SELECT pc.ProductCategoryCode, pc.ProductCategoryName, COUNT(p.ProductCode) AS num_products
    FROM ProductCategory pc
    LEFT JOIN Product p ON pc.ProductCategoryCode = p.ProductCategoryCode
    GROUP BY pc.ProductCategoryCode
''', conn)

,ProductCategoryCode,ProductCategoryName,num_products
0,1,CategoryA,1
1,2,CategoryM,1
2,3,CategoryO,1


### Relationship 4: Product (1) → OrderDetails (\*)

In [15]:
pd.read_sql('''
    SELECT p.ProductCode, p.ProductName, COUNT(od.OrderCode) AS num_order_lines
    FROM Product p
    LEFT JOIN OrderDetails od ON p.ProductCode = od.ProductCode
    GROUP BY p.ProductCode
''', conn)

,ProductCode,ProductName,num_order_lines
0,500,ProductA,1
1,600,ProductM,1
2,700,ProductO,2


`ProductO` appears on **2** separate order lines (order 100 line 3, and order 101 line 1) — proof that one Product can appear across many OrderDetails.

### Relationship 5: Orders (1) → OrderDetails (\*) — the Master/Parent → Details/Child relationship

In [16]:
pd.read_sql('''
    SELECT o.OrderCode, o.OrderDate, COUNT(od.OrderLineCode) AS num_lines
    FROM Orders o
    LEFT JOIN OrderDetails od ON o.OrderCode = od.OrderCode
    GROUP BY o.OrderCode
''', conn)

,OrderCode,OrderDate,num_lines
0,100,1405-05-05,3
1,101,1405-05-06,1


Order 100 has **3** lines, order 101 has **1** — direct proof of `Orders 1 ─── * OrderDetails`. This is the relationship called **Master → Details** or **Parent → Child**: `Orders` is the header (Master/Parent), `OrderDetails` is the set of line items underneath it (Details/Child).

## 6. Reconstructing the Original Flat View With JOIN

To prove nothing was lost in the split, let's rebuild the exact original flat table using `JOIN`s across all six normalized tables:

In [17]:
rebuilt = pd.read_sql('''
    SELECT
        o.OrderCode,
        od.OrderLineCode,
        od.ProductCode,
        p.UnitPrice,
        od.Quantity AS OrderQuantity,
        od.TaxAmount,
        od.Discount,
        p.ProductName,
        c.CustomerCode,
        c.CustomerName,
        e.EmployeeCode,
        e.EmployeeName,
        o.OrderDate,
        pc.ProductCategoryCode,
        pc.ProductCategoryName
    FROM OrderDetails od
    JOIN Orders o           ON od.OrderCode = o.OrderCode
    JOIN Product p          ON od.ProductCode = p.ProductCode
    JOIN Customer c         ON o.CustomerCode = c.CustomerCode
    JOIN Employee e         ON o.EmployeeCode = e.EmployeeCode
    JOIN ProductCategory pc ON p.ProductCategoryCode = pc.ProductCategoryCode
    ORDER BY o.OrderCode, od.OrderLineCode
''', conn)
rebuilt

,OrderCode,OrderLineCode,ProductCode,UnitPrice,OrderQuantity,TaxAmount,Discount,ProductName,CustomerCode,CustomerName,EmployeeCode,EmployeeName,OrderDate,ProductCategoryCode,ProductCategoryName
0,100,1,500,1000,1,0.1,0.01,ProductA,555,Ali Rahmani,666,Behnam Shaygan,1405-05-05,1,CategoryA
1,100,2,600,4000,2,0.0,0.00,ProductM,555,Ali Rahmani,666,Behnam Shaygan,1405-05-05,2,CategoryM
2,100,3,700,80000,1,0.2,0.02,ProductO,555,Ali Rahmani,666,Behnam Shaygan,1405-05-05,3,CategoryO
3,101,1,700,80000,3,0.2,0.02,ProductO,556,Fatemeh Rezaeie,666,Behnam Shaygan,1405-05-06,3,CategoryO


Compare this to the very first cell in this notebook — same data, same values, reconstructed entirely from six small, non-redundant tables instead of one repetitive flat one.

## 7. Final Reference: Primary Keys, Foreign Keys, and Relationships

| Table | Primary Key | Foreign Keys |
|---|---|---|
| Customer | CustomerCode | — |
| Employee | EmployeeCode | — |
| ProductCategory | ProductCategoryCode | — |
| Product | ProductCode | ProductCategoryCode → ProductCategory |
| Orders | OrderCode | CustomerCode → Customer, EmployeeCode → Employee |
| OrderDetails | (OrderCode, OrderLineCode) — Composite | OrderCode → Orders, ProductCode → Product |

**All relationships found, all One-to-Many:**

```text
Customer         1 ─── * Orders
Employee         1 ─── * Orders
Orders           1 ─── * OrderDetails      (Master/Parent → Details/Child)
Product          1 ─── * OrderDetails
ProductCategory  1 ─── * Product
```

Visually:

```text
Customer (1)
     │
     │ *
     ▼
   Orders (1) ────────── (*) OrderDetails (*) ────────── (1) Product
     ▲                                                        │
     │                                                        │ *
Employee (1)                                                  ▼
                                                     ProductCategory (1)
```

## 🔑 Key Takeaways

- The entire example is solved by asking one repeated question: **"What entity does this piece of information actually belong to?"** — every table boundary comes from that answer.
- `OrderDetails` needs a **Composite Primary Key** (`OrderCode` + `OrderLineCode`) because `OrderCode` alone isn't unique — one order can have several lines.
- Every relationship in this specific example turned out to be **One-to-Many** (`1 ─── *`), with the Foreign Key always living on the "many" (Child) side.
- The special case `Orders → OrderDetails` is commonly called **Master → Details** or **Parent → Child** — a header table with a table of line items underneath it, one of the most common patterns in real-world database design.
- A `JOIN` across all six tables reconstructs the original flat view exactly — proving normalization doesn't lose information, it just removes redundancy from how that information is *stored*.

## 1. The second problem: One Flat Table

Everything begins mixed into a single table, one row per order line:

```text
BookCode, BookName, AuthorName, AuthorAddress, AuthorCode, ISBN, BookPrice,
Quantity, OrderCode, OrderDate, Discount, PublisherCode, PublisherName, PublisherAddress
```

Example row:

```text
101, Python, Vahid Ghorbani, Tehran, 500, 577878778..., 1000, 2, 101, 1405-05-05, 0, 444, PublisherA, Iran-Isfahan
```

This single table is actually hiding **four different entities** mixed together: `Book`, `Author`, `Publisher`, and `Order`. The first task, same as before, is separating them.

In [18]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

cur.execute('''
CREATE TABLE Flat (
    BookCode INTEGER,
    BookName TEXT,
    AuthorName TEXT,
    AuthorAddress TEXT,
    AuthorCode INTEGER,
    ISBN TEXT,
    BookPrice INTEGER,
    Quantity INTEGER,
    OrderCode INTEGER,
    OrderDate TEXT,
    Discount REAL,
    PublisherCode INTEGER,
    PublisherName TEXT,
    PublisherAddress TEXT
)
''')
cur.executemany("INSERT INTO Flat VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)", [
    (101, "Python", "Vahid Ghorbani", "Tehran",  500, "577878778XXXX", 1000, 2, 101, "1405-05-05", 0, 444, "PublisherA", "Iran, Isfahan"),
    (101, "Python", "Sara Karimi",    "Shiraz",  501, "577878778XXXX", 1000, 2, 101, "1405-05-05", 0, 444, "PublisherA", "Iran, Isfahan"),
    (102, "SQL 101","Vahid Ghorbani", "Tehran",  500, "412345678XXXX", 1500, 1, 105, "1405-05-10", 0.1, 444, "PublisherA", "Iran, Isfahan"),
])
conn.commit()

pd.read_sql("SELECT * FROM Flat", conn)

,BookCode,BookName,AuthorName,AuthorAddress,AuthorCode,ISBN,BookPrice,Quantity,OrderCode,OrderDate,Discount,PublisherCode,PublisherName,PublisherAddress
0,101,Python,Vahid Ghorbani,Tehran,500,577878778XXXX,1000,2,101,1405-05-05,0.0,444,PublisherA,"Iran, Isfahan"
1,101,Python,Sara Karimi,Shiraz,501,577878778XXXX,1000,2,101,1405-05-05,0.0,444,PublisherA,"Iran, Isfahan"
2,102,SQL 101,Vahid Ghorbani,Tehran,500,412345678XXXX,1500,1,105,1405-05-10,0.1,444,PublisherA,"Iran, Isfahan"


Notice something important already visible in this small sample: `Book 101` ("Python") appears **twice** — once for each of its two authors. That's the Many-to-Many relationship revealing itself even before we start analyzing anything formally.

## 2. Step One: Identify the Key Columns

The natural **candidate keys** — the columns that can uniquely identify one record of each entity — are:

```text
BookCode       → identifies one specific book
AuthorCode     → identifies one specific author
PublisherCode  → identifies one specific publisher
OrderCode      → identifies one specific order
```

### What Makes a Good Primary Key? (Four Properties)

| Property | Meaning |
|---|---|
| **Unique** | No two records can share the same key value — two books can't both be `BookCode = 101` |
| **Not Null** | A key can never be empty — otherwise you couldn't tell which record it belongs to |
| **Not Update** | A key's value generally shouldn't change once assigned, since other tables may already reference it |
| **Narrow Data Type** | Keys should be small and simple (e.g., `INT`), not large — a long string as a primary key bloats indexes and slows down joins |

Let's verify `BookCode` actually satisfies "Unique" and "Not Null" with a direct query, the same way we'd check any real dataset:

In [19]:
pd.read_sql('''
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT BookCode) AS distinct_books,
        SUM(CASE WHEN BookCode IS NULL THEN 1 ELSE 0 END) AS null_books
    FROM Flat
''', conn)

,total_rows,distinct_books,null_books
0,3,2,0


`total_rows` (3) is greater than `distinct_books` (2) — which is expected and fine here, since `BookCode` isn't meant to be unique in this *flat* table (it's not the Flat table's key; `BookCode` will only become a clean Primary Key once it lives in its own dedicated `Books` table).

## 3. Why Isn't `ISBN` the Primary Key?

`ISBN` looks like an obvious candidate — it's a real, globally meaningful identifier for a book. But there's a reason the design instead uses `BookCode` as the Primary Key and keeps `ISBN` as just a regular attribute. This is the distinction between two kinds of keys:

| | Business Key | Surrogate Key |
|---|---|---|
| **Definition** | A key that comes from the real world / business domain, carrying real meaning | An artificial key created purely for database purposes, with no real-world meaning |
| **Examples** | ISBN, National ID, Email, Product Code | `BookCode = 101`, `102`, `103` |
| **In this example** | `ISBN` | `BookCode` |

`ISBN` genuinely identifies a book in the real world — but it's a long string (13 digits, commonly 13 bytes), which conflicts with the "Narrow Data Type" property above. `BookCode` is a small integer that means nothing on its own — it doesn't describe the book, it just labels *which row* this is — but it's small, fast to index, and fast to join on. This is a deliberate design trade-off: keep the meaningful identifier (`ISBN`) as an attribute of the book, and use a lightweight artificial identifier (`BookCode`) as the actual Primary Key.

## 4. `IDENTITY` — Letting the Database Generate Surrogate Keys Automatically

Since a Surrogate Key like `BookCode` has no real-world meaning, there's no reason to type it in by hand — the database can generate it automatically. In SQL Server, this is written as:

```sql
BookCode INT IDENTITY(1,1)
```

This means:

```text
Seed      = 1   (the first value used)
Increment = 1   (how much each new value increases by)
```

So SQL Server automatically produces `1, 2, 3, 4, 5, ...` for every new row, without you specifying `BookCode` in the `INSERT` statement at all:

```sql
INSERT INTO Books (BookName, ISBN, BookPrice, PublisherCode)
VALUES ('Python', '577878778...', 1000, 444);
-- SQL Server assigns BookCode = 1 automatically
```

*(SQLite, used in this notebook, achieves the same effect with `INTEGER PRIMARY KEY AUTOINCREMENT` — same idea, different syntax, shown below.)*

## 5. Separating the Entities

Four straightforward entities, plus one relationship that needs special handling:

```text
Book
Author
Publisher
Order
```

...and the special case: **Book ↔ Author**, which we'll get to after building the simple tables first.

In [20]:
# Publishers — one row per publisher
cur.execute('''
CREATE TABLE Publishers (
    PublisherCode INTEGER PRIMARY KEY,
    PublisherName TEXT NOT NULL,
    PublisherAddress TEXT
)
''')
cur.executemany("INSERT INTO Publishers VALUES (?, ?, ?)", [
    (444, "PublisherA", "Iran, Isfahan"),
])

# Authors — one row per author
cur.execute('''
CREATE TABLE Authors (
    AuthorCode INTEGER PRIMARY KEY,
    AuthorName TEXT NOT NULL,
    AuthorAddress TEXT
)
''')
cur.executemany("INSERT INTO Authors VALUES (?, ?, ?)", [
    (500, "Vahid Ghorbani", "Tehran"),
    (501, "Sara Karimi",    "Shiraz"),
])
conn.commit()
print("Publishers and Authors created")

Publishers and Authors created


In [21]:
# Books — surrogate key (BookCode) auto-generated; ISBN kept as a Business Key attribute
cur.execute('''
CREATE TABLE Books (
    BookCode INTEGER PRIMARY KEY AUTOINCREMENT,  -- Surrogate Key, like IDENTITY(1,1) in SQL Server
    BookName TEXT NOT NULL,
    ISBN TEXT NOT NULL,                            -- Business Key, kept as a regular attribute
    BookPrice INTEGER NOT NULL,
    PublisherCode INTEGER,
    FOREIGN KEY (PublisherCode) REFERENCES Publishers(PublisherCode)
)
''')
# Note: we insert explicit BookCode values here to match the original example's numbering (101, 102)
cur.execute("INSERT INTO Books (BookCode, BookName, ISBN, BookPrice, PublisherCode) VALUES (?,?,?,?,?)",
            (101, "Python", "577878778XXXX", 1000, 444))
cur.execute("INSERT INTO Books (BookCode, BookName, ISBN, BookPrice, PublisherCode) VALUES (?,?,?,?,?)",
            (102, "SQL 101", "412345678XXXX", 1500, 444))
conn.commit()
print("Books created — PublisherCode is a Foreign Key")

Books created — PublisherCode is a Foreign Key


In [22]:
# Orders — one row per order; references Books via a Foreign Key
cur.execute('''
CREATE TABLE Orders (
    OrderCode INTEGER PRIMARY KEY,
    OrderDate TEXT NOT NULL,
    BookCode INTEGER,
    Quantity INTEGER,
    Discount REAL,
    FOREIGN KEY (BookCode) REFERENCES Books(BookCode)
)
''')
cur.executemany("INSERT INTO Orders VALUES (?, ?, ?, ?, ?)", [
    (101, "1405-05-05", 101, 2, 0),
    (105, "1405-05-10", 102, 1, 0.1),
])
conn.commit()
print("Orders created — BookCode is a Foreign Key")

Orders created — BookCode is a Foreign Key


## 6. The Core Problem: Why Do We Need `Book_Author`?

Suppose one book only ever had one author. We could simply put `AuthorCode` directly inside `Books`:

```text
BookCode | BookName | AuthorCode
101      | Python   | 500
```

No problem yet. But in the real world:

- One book can have **multiple** authors: `Book A → Author 1, Author 2`
- One author can write **multiple** books: `Author 1 → Book A, Book B, Book C`

This is a genuine **Many-to-Many** relationship. Trying to force it directly into `Books` breaks down immediately — either you duplicate the whole book row per author:

```text
101 | Python | 500
101 | Python | 501        <-- BookName repeated, this is redundancy again
```

or you try to cram multiple authors into one field:

```text
101 | Python | 500,501    <-- not atomic — breaks 1NF directly
```

Neither is acceptable. The fix, exactly like the Students↔Courses case from Session 6, is a **junction (bridge) table** sitting between the two entities.

In [23]:
# Book_Author — the junction table resolving the Many-to-Many relationship
cur.execute('''
CREATE TABLE Book_Author (
    BookCode INTEGER,
    AuthorCode INTEGER,
    PRIMARY KEY (BookCode, AuthorCode),
    FOREIGN KEY (BookCode) REFERENCES Books(BookCode),
    FOREIGN KEY (AuthorCode) REFERENCES Authors(AuthorCode)
)
''')
cur.executemany("INSERT INTO Book_Author VALUES (?, ?)", [
    (101, 500),   # Python  -> Vahid Ghorbani
    (101, 501),   # Python  -> Sara Karimi
    (102, 500),   # SQL 101 -> Vahid Ghorbani
])
conn.commit()

pd.read_sql("SELECT * FROM Book_Author", conn)

,BookCode,AuthorCode
0,101,500
1,101,501
2,102,500


### Why `(BookCode, AuthorCode)` Has to Be a Composite Key

Neither column is unique on its own:

```text
BookCode alone:    101, 101, 102   -> repeats
AuthorCode alone:  500, 501, 500   -> repeats
```

But the **pair** never repeats — `(101,500)`, `(101,501)`, `(102,500)` are each unique. That combination is the table's Composite Primary Key, confirmed directly by query:

In [24]:
pd.read_sql('''
    SELECT BookCode, AuthorCode, COUNT(*) AS times_seen
    FROM Book_Author
    GROUP BY BookCode, AuthorCode
''', conn)

,BookCode,AuthorCode,times_seen
0,101,500,1
1,101,501,1
2,102,500,1


Every combination appears exactly once — proving the pair is a valid unique key, even though neither column is unique by itself.

## 7. How `Book_Author` Turns One Many-to-Many Into Two One-to-Manys

```text
Books  1 ─── * Book_Author * ─── 1  Authors
```

In other words: one Book can have many rows in `Book_Author` (one per author), and one Author can have many rows in `Book_Author` (one per book) — but each individual row in `Book_Author` connects exactly one Book to exactly one Author. The Many-to-Many relationship never has to be represented directly; it's expressed as two ordinary One-to-Many relationships pointing at the same junction table.

In [25]:
print("Authors per book (proving Books 1 --- * Book_Author):")
display(pd.read_sql('''
    SELECT b.BookCode, b.BookName, COUNT(ba.AuthorCode) AS num_authors
    FROM Books b
    LEFT JOIN Book_Author ba ON b.BookCode = ba.BookCode
    GROUP BY b.BookCode
''', conn))

print("\nBooks per author (proving Authors 1 --- * Book_Author):")
display(pd.read_sql('''
    SELECT a.AuthorCode, a.AuthorName, COUNT(ba.BookCode) AS num_books
    FROM Authors a
    LEFT JOIN Book_Author ba ON a.AuthorCode = ba.AuthorCode
    GROUP BY a.AuthorCode
''', conn))

Authors per book (proving Books 1 --- * Book_Author):


,BookCode,BookName,num_authors
0,101,Python,2
1,102,SQL 101,1



Books per author (proving Authors 1 --- * Book_Author):


,AuthorCode,AuthorName,num_books
0,500,Vahid Ghorbani,2
1,501,Sara Karimi,1


`Python` has **2** authors, and `Vahid Ghorbani` has written **2** books — both counts exceed 1, which is exactly what proves this is genuinely Many-to-Many, unlike every relationship in the previous Orders example, which was One-to-Many in only one direction.

## 8. The Two Simple One-to-Many Relationships in This Schema

### Publisher → Books

In [26]:
pd.read_sql('''
    SELECT p.PublisherCode, p.PublisherName, COUNT(b.BookCode) AS num_books
    FROM Publishers p
    LEFT JOIN Books b ON p.PublisherCode = b.PublisherCode
    GROUP BY p.PublisherCode
''', conn)

,PublisherCode,PublisherName,num_books
0,444,PublisherA,2


One publisher, two books → `Publishers 1 ─── * Books`.

### Books → Orders

In [27]:
pd.read_sql('''
    SELECT b.BookCode, b.BookName, COUNT(o.OrderCode) AS num_orders
    FROM Books b
    LEFT JOIN Orders o ON b.BookCode = o.BookCode
    GROUP BY b.BookCode
''', conn)

,BookCode,BookName,num_orders
0,101,Python,1
1,102,SQL 101,1


Each book here has one order in this small sample, but the design allows any book to appear on many orders over time — `Books 1 ─── * Orders`.

## 9. Reconstructing the Original View With JOIN

Pulling every table back together to confirm nothing was lost:

In [28]:
pd.read_sql('''
    SELECT
        b.BookCode, b.BookName, a.AuthorName, a.AuthorAddress,
        b.ISBN, b.BookPrice, o.Quantity, o.OrderCode, o.OrderDate,
        o.Discount, p.PublisherCode, p.PublisherName, p.PublisherAddress
    FROM Books b
    JOIN Book_Author ba ON b.BookCode = ba.BookCode
    JOIN Authors a      ON ba.AuthorCode = a.AuthorCode
    JOIN Publishers p   ON b.PublisherCode = p.PublisherCode
    LEFT JOIN Orders o  ON b.BookCode = o.BookCode
    ORDER BY b.BookCode, a.AuthorCode
''', conn)

,BookCode,BookName,AuthorName,AuthorAddress,ISBN,BookPrice,Quantity,OrderCode,OrderDate,Discount,PublisherCode,PublisherName,PublisherAddress
0,101,Python,Vahid Ghorbani,Tehran,577878778XXXX,1000,2,101,1405-05-05,0.0,444,PublisherA,"Iran, Isfahan"
1,101,Python,Sara Karimi,Shiraz,577878778XXXX,1000,2,101,1405-05-05,0.0,444,PublisherA,"Iran, Isfahan"
2,102,SQL 101,Vahid Ghorbani,Tehran,412345678XXXX,1500,1,105,1405-05-10,0.1,444,PublisherA,"Iran, Isfahan"


Notice `Python` (BookCode 101) now correctly appears once per author via the join — exactly matching the original flat table's structure, but built from five small, non-redundant tables instead of one repetitive one.